# Build ML-Ready Datasets from Raw JSON

This notebook turns the **raw scraper JSON files** (from your `conjugate-media` zip) into two clean tables for ML:

| Output file | What it is | Used by |
|-------------|------------|---------|
| `model_A_trending.csv` | Trending posts/videos from all platforms | **Model A** (what is trending?) |
| `model_B_audience_posts.csv` | Creator’s own posts only (no trending) | **Model B** (what audience do I have?) |

### Engineered features we will create
- `breakout_log_ratio` = ln((views+1) / (subscribers+1))
- `quality_reaction_rate` = (likes + comments + all_reactions) / views
- `virality_index` = views + 15×likes + 50×comments + 100×shares
- `engagement_score` = composite 0–100 score (good ML target)
- `content_type`, `duration_sec`, `is_vertical_reel`, `is_short_form`
- `post_hour`, `post_dayofweek`, `is_weekend`
- `title`, `description`, `category`, `tags`

**No re-scraping needed** — we only read the JSON already in `server/data/`.


## 1. Setup — imports and paths

We only need `pandas`, `numpy`, and the standard library.


In [1]:
import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# Where the raw JSON files live
DATA = Path('C:/Users/saura/OneDrive/Desktop/projects/sem7/EngageIQfinal/EngageIQ/conjugate-media/server/data')
assert DATA.exists(), f'Not found: {DATA.resolve()} — unzip the project first'

OUT = DATA / 'ml_ready'
OUT.mkdir(parents=True, exist_ok=True)

print('DATA :', DATA.resolve())
print('OUT  :', OUT.resolve())
print('\nJSON files available:')
for p in sorted(DATA.glob('*.json')):
    print(f'  {p.name:60s}  {p.stat().st_size/1024:7.1f} KB')


DATA : C:\Users\saura\OneDrive\Desktop\projects\sem7\EngageIQfinal\EngageIQ\conjugate-media\server\data
OUT  : C:\Users\saura\OneDrive\Desktop\projects\sem7\EngageIQfinal\EngageIQ\conjugate-media\server\data\ml_ready

JSON files available:
  conjugate_users.json                                              4.6 KB
  dataset_facebook-posts-scraper_2026-10-11_02-36-53-748.json     221.0 KB
  dataset_facebook-trending-scraper_2026-10-11_13-40-00-000.json     15.8 KB
  dataset_instagram-profile-scraper_2026-10-10_12-32-43-860.json     30.5 KB
  dataset_instagram-trending-scraper_2026-10-11_13-34-15-622.json     98.3 KB
  dataset_linkedin-profile-posts_2026-10-11_02-59-19-515.json     495.5 KB
  dataset_linkedin-trending-scraper_2026-10-11_13-45-00-000.json     16.9 KB
  dataset_twitter-trends_2026-10-11_13-24-06-583.json              13.2 KB
  dataset_twitter-user-tweets_2026-10-11_02-24-01-054.json        408.0 KB
  dataset_youtube-channel-scraper_2026-10-10_15-39-13-562.json    543.8 KB
 

## 2. Helper functions

These small functions clean messy real-world fields so every platform looks the same.


### 2.1 Load JSON & safe numbers


In [2]:
def load_json(name):
    """Read a JSON file from the data folder."""
    with open(DATA / name, encoding='utf-8') as f:
        return json.load(f)

def safe_num(x, default=0.0):
    """Convert anything to float; return default if it fails."""
    if x is None:
        return default
    try:
        v = float(x)
        if isinstance(v, float) and math.isnan(v):
            return default
        return v
    except (TypeError, ValueError):
        return default

print('Helpers ready: load_json, safe_num')


Helpers ready: load_json, safe_num


### 2.2 Parse duration (`"5:57"` → seconds) and posting time


In [3]:
def parse_duration_sec(val):
    """'5:57' or '1:02:30' or 357 → seconds (float)."""
    if val is None:
        return np.nan
    if isinstance(val, (int, float)):
        return float(val) if not (isinstance(val, float) and math.isnan(val)) else np.nan
    s = str(val).strip()
    if not s:
        return np.nan
    # ISO 8601 like PT1H2M3S
    m = re.match(r'^PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?$', s, re.I)
    if m:
        h, mi, sec = (int(x) if x else 0 for x in m.groups())
        return h * 3600 + mi * 60 + sec
    parts = s.split(':')
    try:
        parts = [int(p) for p in parts]
    except ValueError:
        try:
            return float(s)
        except ValueError:
            return np.nan
    if len(parts) == 3:
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    if len(parts) == 2:
        return parts[0] * 60 + parts[1]
    return float(parts[0]) if parts else np.nan


def parse_time(val):
    """Return (iso_string, hour 0-23, dayofweek 0=Mon, is_weekend 0/1)."""
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return None, np.nan, np.nan, np.nan
    # Unix timestamp (seconds or milliseconds)
    if isinstance(val, (int, float)):
        ts = float(val)
        if ts > 1e12:
            ts /= 1000.0
        try:
            dt = datetime.fromtimestamp(ts, tz=timezone.utc)
        except (ValueError, OSError, OverflowError):
            return None, np.nan, np.nan, np.nan
        return dt.isoformat(), dt.hour, dt.weekday(), int(dt.weekday() >= 5)
    s = str(val).strip()
    if not s:
        return None, np.nan, np.nan, np.nan
    for fmt in (
        '%Y-%m-%dT%H:%M:%S.%fZ',
        '%Y-%m-%dT%H:%M:%SZ',
        '%Y-%m-%dT%H:%M:%S%z',
        '%Y-%m-%dT%H:%M:%S',
        '%Y-%m-%d %H:%M:%S',
        '%a %b %d %H:%M:%S %z %Y',
    ):
        try:
            dt = datetime.strptime(s.replace('+0000', '+00:00'), fmt)
            if dt.tzinfo is None:
                dt = dt.replace(tzinfo=timezone.utc)
            return dt.isoformat(), dt.hour, dt.weekday(), int(dt.weekday() >= 5)
        except ValueError:
            continue
    try:
        dt = datetime.fromisoformat(s.replace('Z', '+00:00'))
        return dt.isoformat(), dt.hour, dt.weekday(), int(dt.weekday() >= 5)
    except ValueError:
        return None, np.nan, np.nan, np.nan

# Quick test
print('duration 5:57 →', parse_duration_sec('5:57'), 'sec')
print('time parse   →', parse_time('2026-10-10T03:47:32Z')[:3])


duration 5:57 → 357 sec
time parse   → ('2026-10-10T03:47:32+00:00', 3, 5)


### 2.3 Content type, reel / short-form flags, tags


In [4]:
def infer_content_type(platform, post_type=None, type_=None, duration_sec=None,
                       media_url=None, video_url=None, attachments=None, product_type=None):
    """Normalize to: video | image | text | reel | short | carousel | unknown"""
    raw = ' '.join(str(x).lower() for x in (post_type, type_, product_type) if x)
    if 'reel' in raw:
        return 'reel'
    if 'short' in raw:
        return 'short'
    if 'carousel' in raw or 'sidecar' in raw:
        return 'carousel'
    if 'video' in raw or 'feed_video' in raw or raw in ('movie', 'clip'):
        return 'video'
    if 'image' in raw or 'photo' in raw or 'picture' in raw or raw == 'static':
        return 'image'
    if video_url or (media_url and any(e in str(media_url).lower() for e in ('.mp4', '.mov', 'video'))):
        return 'video'
    if duration_sec is not None and not (isinstance(duration_sec, float) and math.isnan(duration_sec)) and duration_sec > 0:
        return 'video'
    if attachments and isinstance(attachments, list) and attachments:
        t = str(attachments[0].get('type', '') if isinstance(attachments[0], dict) else attachments[0]).lower()
        if 'video' in t:
            return 'video'
        if 'image' in t or 'photo' in t:
            return 'image'
    if media_url:
        return 'image'
    if raw in ('text', 'tweet', 'status', 'post') or not raw:
        return 'text'
    return 'unknown' if not raw else raw


def is_vertical_reel(content_type, post_type=None, product_type=None, platform=None):
    raw = ' '.join(str(x).lower() for x in (content_type, post_type, product_type, platform) if x)
    if 'reel' in raw or 'short' in raw:
        return 1
    if platform == 'instagram' and content_type in ('video', 'reel'):
        return 1
    if platform == 'youtube' and content_type in ('short', 'reel'):
        return 1
    return 0


def is_short_form(content_type, duration_sec, platform=None):
    if content_type in ('reel', 'short'):
        return 1
    if duration_sec is not None and not (isinstance(duration_sec, float) and math.isnan(duration_sec)):
        if duration_sec <= 60:
            return 1
        if duration_sec <= 180 and platform in ('instagram', 'tiktok', 'youtube'):
            return 1
    return 0


def tags_to_str(tags):
    if tags is None:
        return ''
    if isinstance(tags, list):
        return ' | '.join(str(t) for t in tags if t)
    return str(tags)


def tags_count(tags):
    if tags is None:
        return 0
    if isinstance(tags, list):
        return len(tags)
    if isinstance(tags, str) and tags.strip():
        return len(re.split(r'[,|]', tags))
    return 0


def all_reactions_count(reactions_obj, likes_fallback=0):
    """Sum like+love+wow+... if Facebook-style dict exists."""
    if isinstance(reactions_obj, dict):
        return sum(safe_num(v) for v in reactions_obj.values())
    return safe_num(likes_fallback)

print('Content helpers ready')


Content helpers ready


### 2.4 Engagement metrics (the important formulas)

\\[
\begin{align*}
\text{breakout\_log\_ratio} &= \ln\frac{\text{views}+1}{\text{subscribers}+1} \\
\text{quality\_reaction\_rate} &= \frac{\text{likes}+\text{comments}+\text{all\_reactions}}{\text{views}} \\
\text{virality\_index} &= \text{views} + 15\times\text{likes} + 50\times\text{comments} + 100\times\text{shares} \\
\text{engagement\_score} &= \text{composite 0–100 (target for ML)}
\end{align*}
\\]


In [5]:
def compute_metrics(row):
    """Add all engineered engagement columns to a row dict."""
    views = safe_num(row.get('views'))
    likes = safe_num(row.get('likes'))
    comments = safe_num(row.get('comments'))
    shares = safe_num(row.get('shares'))
    reactions_all = safe_num(row.get('reactions_all'), likes)
    subs = safe_num(row.get('subscribers'), 0)

    qrr = (likes + comments + reactions_all) / views if views > 0 else np.nan
    breakout = math.log((views + 1) / (subs + 1))
    virality = views + 15 * likes + 50 * comments + 100 * shares

    # engagement_rate (%)
    er = 100.0 * (likes + comments + shares) / views if views > 0 else 0.0

    # Soft-normalized pieces → engagement_score 0..100
    er_s  = min(er / 15.0, 1.0)          # ~15% ER = max
    qrr_s = min((qrr if qrr == qrr else 0) / 0.20, 1.0)
    vir_s = min(math.log1p(virality) / 20.0, 1.0)
    brk_s = min(max((breakout + 2) / 8.0, 0), 1.0)
    engagement_score = 100.0 * (0.35 * er_s + 0.25 * qrr_s + 0.25 * vir_s + 0.15 * brk_s)

    return {
        'views': views,
        'likes': likes,
        'comments': comments,
        'shares': shares,
        'reactions_all': reactions_all,
        'subscribers': subs,
        'engagement_rate': er,
        'quality_reaction_rate': qrr,
        'breakout_log_ratio': breakout,
        'virality_index': virality,
        'engagement_score': engagement_score,
    }

# Demo
demo = compute_metrics({'views': 1_000_000, 'likes': 50_000, 'comments': 2_000, 'shares': 500, 'reactions_all': 50_000, 'subscribers': 100_000})
print({k: round(v, 3) if isinstance(v, float) else v for k, v in demo.items()})


{'views': 1000000.0, 'likes': 50000.0, 'comments': 2000.0, 'shares': 500.0, 'reactions_all': 50000.0, 'subscribers': 100000.0, 'engagement_rate': 5.25, 'quality_reaction_rate': 0.102, 'breakout_log_ratio': 2.303, 'virality_index': 1900000.0, 'engagement_score': 51.139}


## 3. Subscriber / follower map (needed for breakout ratio)

Model B needs **subscribers per platform** so  
`breakout_log_ratio = ln((views+1)/(subs+1))` is meaningful.
We read them from `default_seed.json` and `live_store.json`.


In [6]:
seed = load_json('default_seed.json')
live = load_json('live_store.json')

channel_subs = {}
user_subs = {}

for plat, ch in (live.get('default_user') or {}).get('channels', {}).items():
    channel_subs[plat.lower()] = safe_num(ch.get('subscribers') or ch.get('followers'))

for u in seed.get('users', []):
    plat = (u.get('platform') or '').lower()
    user_subs[(u.get('id'), plat)] = safe_num(u.get('followers'))
    user_subs[(u.get('username'), plat)] = safe_num(u.get('followers'))
    channel_subs.setdefault(plat, safe_num(u.get('followers')))

def lookup_subs(platform, user_id=None, username=None):
    platform = (platform or '').lower()
    if user_id and (user_id, platform) in user_subs:
        return user_subs[(user_id, platform)]
    if username and (username, platform) in user_subs:
        return user_subs[(username, platform)]
    return channel_subs.get(platform, 0.0)

print('Subscribers by platform:')
for p, s in sorted(channel_subs.items()):
    print(f'  {p:12s}  {s:,.0f}')


Subscribers by platform:
  facebook      1,250,000
  instagram     3,662,163
  linkedin      485,000
  twitter       650,000
  youtube       8,630,000


---
## 4. MODEL A — Build trending table

We walk each **trending** JSON file and convert every item into one common row.


### 4.1 YouTube trending

In [7]:
rows_A = []

for it in load_json('dataset_youtube-trending-scraper_2026-10-11_13-13-46-735.json'):
    dur = parse_duration_sec(it.get('duration'))
    pub_iso, hour, dow, weekend = parse_time(it.get('published_at'))
    ctype = infer_content_type('youtube', duration_sec=dur, video_url=it.get('video_url'))
    if dur == dur and dur <= 60:  # not NaN and short
        ctype = 'short'
    tags = it.get('tags') or []
    row = {
        'platform': 'youtube',
        'content_id': it.get('video_id'),
        'title': it.get('title') or '',
        'description': (it.get('description') or '')[:2000],
        'category': str(it.get('video_category_id') or it.get('category') or ''),
        'tags': tags_to_str(tags),
        'tags_count': tags_count(tags),
        'author': it.get('channel_title'),
        'url': it.get('video_url'),
        'content_type': ctype,
        'duration_sec': dur,
        'published_at': pub_iso,
        'post_hour': hour,
        'post_dayofweek': dow,
        'is_weekend': weekend,
        'is_vertical_reel': is_vertical_reel(ctype, platform='youtube'),
        'is_short_form': is_short_form(ctype, dur, 'youtube'),
        'views': safe_num(it.get('view_count')),
        'likes': safe_num(it.get('like_count')),
        'comments': safe_num(it.get('comment_count')),
        'shares': 0.0,
        'reactions_all': safe_num(it.get('like_count')),
        'subscribers': 0.0,  # channel size not in this scrape
        'region': it.get('region_code'),
        'source': 'trending',
    }
    row.update(compute_metrics(row))
    rows_A.append(row)

print(f'YouTube trending rows so far: {len(rows_A)}')
print('Sample title:', rows_A[0]['title'][:60])
print('engagement_score:', round(rows_A[0]['engagement_score'], 2))


YouTube trending rows so far: 48
Sample title: 295 (MUSIC VIDEO) SIDHU MOOSE WALA
engagement_score: 63.63


### 4.2 Instagram trending

In [8]:
for it in load_json('dataset_instagram-trending-scraper_2026-10-11_13-34-15-622.json'):
    dur = parse_duration_sec(it.get('video_duration') or it.get('duration'))
    pub_iso, hour, dow, weekend = parse_time(it.get('published_at') or it.get('caption_created_at'))
    ctype = infer_content_type('instagram', post_type=it.get('post_type'),
                              product_type=it.get('product_type'), duration_sec=dur)
    tags = it.get('hashtags') or []
    likes = safe_num(it.get('like_count'))
    row = {
        'platform': 'instagram',
        'content_id': str(it.get('post_id') or it.get('shortcode') or ''),
        'title': (it.get('caption') or '')[:200],
        'description': (it.get('caption') or '')[:2000],
        'category': it.get('explore_topic') or it.get('post_type') or '',
        'tags': tags_to_str(tags),
        'tags_count': tags_count(tags),
        'author': it.get('owner_username'),
        'url': it.get('post_url'),
        'content_type': ctype,
        'duration_sec': dur,
        'published_at': pub_iso,
        'post_hour': hour,
        'post_dayofweek': dow,
        'is_weekend': weekend,
        'is_vertical_reel': is_vertical_reel(ctype, it.get('post_type'), it.get('product_type'), 'instagram'),
        'is_short_form': is_short_form(ctype, dur, 'instagram'),
        'views': safe_num(it.get('view_count') or it.get('play_count')),
        'likes': likes,
        'comments': safe_num(it.get('comment_count')),
        'shares': 0.0,
        'reactions_all': likes,
        'subscribers': 0.0,
        'region': it.get('requested_country'),
        'source': 'trending',
    }
    row.update(compute_metrics(row))
    rows_A.append(row)

print(f'After Instagram: {len(rows_A)} rows')


After Instagram: 84 rows


### 4.3 Facebook trending

In [9]:
for it in load_json('dataset_facebook-trending-scraper_2026-10-11_13-40-00-000.json'):
    dur = parse_duration_sec(it.get('duration'))
    pub_iso, hour, dow, weekend = parse_time(it.get('timestamp'))
    author = it.get('author')
    author_name = author.get('name') if isinstance(author, dict) else author
    tags = it.get('tags') or []
    reactions = it.get('reactions') or {}
    likes = safe_num(it.get('reactionsCount') or reactions.get('like'))
    r_all = all_reactions_count(reactions, likes)
    ctype = infer_content_type('facebook', duration_sec=dur, attachments=it.get('attachments'))
    row = {
        'platform': 'facebook',
        'content_id': str(it.get('postId') or ''),
        'title': (it.get('title') or it.get('postText') or '')[:200],
        'description': (it.get('postText') or it.get('title') or '')[:2000],
        'category': it.get('category') or '',
        'tags': tags_to_str(tags),
        'tags_count': tags_count(tags),
        'author': author_name,
        'url': it.get('url'),
        'content_type': ctype,
        'duration_sec': dur,
        'published_at': pub_iso,
        'post_hour': hour,
        'post_dayofweek': dow,
        'is_weekend': weekend,
        'is_vertical_reel': is_vertical_reel(ctype, platform='facebook'),
        'is_short_form': is_short_form(ctype, dur, 'facebook'),
        'views': safe_num(it.get('view_count')),
        'likes': likes,
        'comments': safe_num(it.get('commentsCount')),
        'shares': safe_num(it.get('sharesCount')),
        'reactions_all': r_all,
        'subscribers': 0.0,
        'region': None,
        'source': 'trending',
    }
    row.update(compute_metrics(row))
    rows_A.append(row)

print(f'After Facebook: {len(rows_A)} rows')


After Facebook: 94 rows


### 4.4 LinkedIn trending

In [10]:
for it in load_json('dataset_linkedin-trending-scraper_2026-10-11_13-45-00-000.json'):
    dur = parse_duration_sec(it.get('duration'))
    posted = it.get('postedAt')
    if isinstance(posted, dict):
        posted = posted.get('date') or posted.get('timestamp')
    pub_iso, hour, dow, weekend = parse_time(posted)
    eng = it.get('engagement') if isinstance(it.get('engagement'), dict) else {}
    likes = safe_num(eng.get('numLikes') or eng.get('likes'))
    comments = safe_num(eng.get('numComments') or eng.get('comments'))
    shares = safe_num(eng.get('numShares') or eng.get('shares') or eng.get('numReposts'))
    views = safe_num(eng.get('numViews') or eng.get('views'))
    author = it.get('author')
    author_name = author.get('name') if isinstance(author, dict) else author
    tags = it.get('tags') or []
    post_video = it.get('postVideo')
    video_url = post_video.get('url') if isinstance(post_video, dict) else None
    ctype = infer_content_type('linkedin', post_type=it.get('type'), duration_sec=dur, video_url=video_url)
    row = {
        'platform': 'linkedin',
        'content_id': str(it.get('id') or ''),
        'title': (it.get('title') or (it.get('content') or '')[:120]),
        'description': (it.get('content') or it.get('title') or '')[:2000],
        'category': it.get('category') or it.get('type') or '',
        'tags': tags_to_str(tags),
        'tags_count': tags_count(tags),
        'author': author_name,
        'url': it.get('linkedinUrl'),
        'content_type': ctype,
        'duration_sec': dur,
        'published_at': pub_iso,
        'post_hour': hour,
        'post_dayofweek': dow,
        'is_weekend': weekend,
        'is_vertical_reel': is_vertical_reel(ctype, platform='linkedin'),
        'is_short_form': is_short_form(ctype, dur, 'linkedin'),
        'views': views,
        'likes': likes,
        'comments': comments,
        'shares': shares,
        'reactions_all': likes,
        'subscribers': 0.0,
        'region': None,
        'source': 'trending',
    }
    row.update(compute_metrics(row))
    rows_A.append(row)

print(f'After LinkedIn: {len(rows_A)} rows')


After LinkedIn: 103 rows


### 4.5 Twitter / X trends (topic + volume)

In [11]:
for it in load_json('dataset_twitter-trends_2026-10-11_13-24-06-583.json'):
    vol = it.get('tweets')
    if isinstance(vol, str):
        vol = vol.replace(',', '').upper()
        mult = 1
        if vol.endswith('K'):
            mult, vol = 1_000, vol[:-1]
        elif vol.endswith('M'):
            mult, vol = 1_000_000, vol[:-1]
        try:
            vol = float(vol) * mult
        except ValueError:
            vol = 0.0
    else:
        vol = safe_num(vol)
    pub_iso, hour, dow, weekend = parse_time(it.get('scraped_at') or it.get('date'))
    row = {
        'platform': 'twitter',
        'content_id': f"trend_{it.get('position')}_{it.get('trend')}",
        'title': it.get('trend') or '',
        'description': '',
        'category': 'trend',
        'tags': it.get('trend') or '',
        'tags_count': 1 if it.get('trend') else 0,
        'author': None,
        'url': None,
        'content_type': 'text',
        'duration_sec': np.nan,
        'published_at': pub_iso,
        'post_hour': hour,
        'post_dayofweek': dow,
        'is_weekend': weekend,
        'is_vertical_reel': 0,
        'is_short_form': 1,
        'views': vol,          # proxy: tweet volume
        'likes': 0.0,
        'comments': 0.0,
        'shares': 0.0,
        'reactions_all': 0.0,
        'subscribers': 0.0,
        'region': it.get('country'),
        'source': 'trending',
        'tweet_volume': vol,
        'trend_position': it.get('position'),
    }
    row.update(compute_metrics(row))
    rows_A.append(row)

df_A = pd.DataFrame(rows_A)
print('Model A complete:', df_A.shape)
print(df_A['platform'].value_counts())
print()
print(df_A[['platform', 'title', 'content_type', 'views', 'engagement_score']].head(3))


Model A complete: (153, 32)
platform
twitter      50
youtube      48
instagram    36
facebook     10
linkedin      9
Name: count, dtype: int64

  platform                                              title content_type  \
0  youtube                 295 (MUSIC VIDEO) SIDHU MOOSE WALA        video   
1  youtube  Ranabaali Official Trailer (Telugu) | Vijay De...        video   
2  youtube  MY FIRST DAY AT MY NEW SHOWROOM! | GTA GAMEPLA...        video   

        views  engagement_score  
0  12183039.0         63.627491  
1  16330199.0         40.815721  
2   4825281.0         65.699304  


---
## 5. MODEL B — Build audience / user-posts table

**Only creator posts** (seed store + profile scrapers).  
No trending rows. Includes real **subscriber counts** per platform.


### 5.1 Posts from `default_seed.json` (multi-platform creator content)

In [12]:
rows_B = []

for p in seed.get('posts', []):
    platform = (p.get('platform') or 'unknown').lower()
    metrics = p.get('metrics') or {}
    views = safe_num(p.get('views') or metrics.get('views') or metrics.get('reach'))
    likes = safe_num(p.get('likes') or metrics.get('likes'))
    comments = safe_num(p.get('comments') or metrics.get('comments'))
    shares = safe_num(p.get('shares') or metrics.get('shares'))
    dur = parse_duration_sec(p.get('duration'))
    pub_iso, hour, dow, weekend = parse_time(p.get('publishedAt') or p.get('createdAt'))
    ctype = infer_content_type(platform, post_type=p.get('postType'), type_=p.get('type'),
                              duration_sec=dur, media_url=p.get('mediaUrl'), video_url=p.get('videoUrl'))
    subs = lookup_subs(platform, user_id=p.get('userId'), username=p.get('username'))
    title = p.get('title') or ''
    caption = p.get('caption') or p.get('content') or ''
    row = {
        'platform': platform,
        'content_id': p.get('id'),
        'user_id': p.get('userId'),
        'username': p.get('username'),
        'author': p.get('author') or p.get('username'),
        'title': title[:300],
        'description': caption[:2000],
        'category': p.get('postType') or p.get('type') or '',
        'tags': '',
        'tags_count': 0,
        'url': p.get('mediaUrl') or p.get('videoUrl'),
        'content_type': ctype,
        'duration_sec': dur,
        'published_at': pub_iso,
        'post_hour': hour,
        'post_dayofweek': dow,
        'is_weekend': weekend,
        'is_vertical_reel': is_vertical_reel(ctype, p.get('postType'), platform=platform),
        'is_short_form': is_short_form(ctype, dur, platform),
        'views': views,
        'likes': likes,
        'comments': comments,
        'shares': shares,
        'reactions_all': likes,
        'subscribers': subs,
        'source': 'user_posts',
    }
    row.update(compute_metrics(row))
    rows_B.append(row)

print(f'Seed posts: {len(rows_B)}')
print(pd.Series([r['platform'] for r in rows_B]).value_counts())


Seed posts: 744
twitter      394
linkedin     138
facebook     100
youtube      100
instagram     12
Name: count, dtype: int64


### 5.2 Facebook profile posts

In [13]:
for it in load_json('dataset_facebook-posts-scraper_2026-10-11_02-36-53-748.json'):
    author = it.get('author') or {}
    author_name = author.get('name') if isinstance(author, dict) else author
    reactions = it.get('reactions') or {}
    likes = safe_num(it.get('reactionsCount') or reactions.get('like'))
    r_all = all_reactions_count(reactions, likes)
    dur = parse_duration_sec(it.get('duration'))
    pub_iso, hour, dow, weekend = parse_time(it.get('timestamp'))
    ctype = infer_content_type('facebook', duration_sec=dur, attachments=it.get('attachments'))
    text = it.get('postText') or it.get('title') or ''
    row = {
        'platform': 'facebook',
        'content_id': str(it.get('postId') or ''),
        'user_id': author.get('id') if isinstance(author, dict) else None,
        'username': None,
        'author': author_name,
        'title': text[:200],
        'description': text[:2000],
        'category': '',
        'tags': '',
        'tags_count': 0,
        'url': it.get('url'),
        'content_type': ctype,
        'duration_sec': dur,
        'published_at': pub_iso,
        'post_hour': hour,
        'post_dayofweek': dow,
        'is_weekend': weekend,
        'is_vertical_reel': is_vertical_reel(ctype, platform='facebook'),
        'is_short_form': is_short_form(ctype, dur, 'facebook'),
        'views': safe_num(it.get('view_count') or it.get('video_view_count')),
        'likes': likes,
        'comments': safe_num(it.get('commentsCount')),
        'shares': safe_num(it.get('sharesCount')),
        'reactions_all': r_all,
        'subscribers': lookup_subs('facebook'),
        'source': 'user_posts',
    }
    row.update(compute_metrics(row))
    rows_B.append(row)

print(f'After Facebook posts: {len(rows_B)}')


After Facebook posts: 844


### 5.3 LinkedIn profile posts

In [14]:
for it in load_json('dataset_linkedin-profile-posts_2026-10-11_02-59-19-515.json'):
    eng = it.get('engagement') if isinstance(it.get('engagement'), dict) else {}
    likes = safe_num(eng.get('numLikes') or eng.get('likes'))
    comments = safe_num(eng.get('numComments') or eng.get('comments'))
    shares = safe_num(eng.get('numShares') or eng.get('shares') or eng.get('numReposts'))
    views = safe_num(eng.get('numViews') or eng.get('views'))
    author = it.get('author') or {}
    author_name = author.get('name') if isinstance(author, dict) else author
    posted = it.get('postedAt')
    if isinstance(posted, dict):
        posted = posted.get('date') or posted.get('timestamp')
    pub_iso, hour, dow, weekend = parse_time(posted)
    text = it.get('content') or ''
    ctype = infer_content_type('linkedin', post_type=it.get('type'))
    row = {
        'platform': 'linkedin',
        'content_id': str(it.get('id') or ''),
        'user_id': None,
        'username': None,
        'author': author_name,
        'title': text[:150],
        'description': text[:2000],
        'category': it.get('type') or '',
        'tags': '',
        'tags_count': 0,
        'url': it.get('linkedinUrl'),
        'content_type': ctype,
        'duration_sec': np.nan,
        'published_at': pub_iso,
        'post_hour': hour,
        'post_dayofweek': dow,
        'is_weekend': weekend,
        'is_vertical_reel': 0,
        'is_short_form': 0,
        'views': views,
        'likes': likes,
        'comments': comments,
        'shares': shares,
        'reactions_all': likes,
        'subscribers': lookup_subs('linkedin'),
        'source': 'user_posts',
    }
    row.update(compute_metrics(row))
    rows_B.append(row)

print(f'After LinkedIn posts: {len(rows_B)}')


After LinkedIn posts: 982


### 5.4 Twitter / X user tweets

In [15]:
for it in load_json('dataset_twitter-user-tweets_2026-10-11_02-24-01-054.json'):
    views = safe_num(it.get('views'))
    likes = safe_num(it.get('favorites') or it.get('likes'))
    comments = safe_num(it.get('replies'))
    shares = safe_num(it.get('retweets'))
    pub_iso, hour, dow, weekend = parse_time(it.get('created_at'))
    text = it.get('text') or ''
    media = it.get('media')
    ctype = 'text'
    if media:
        ctype = 'image'
        if isinstance(media, list):
            for m in media:
                if isinstance(m, dict) and 'video' in str(m.get('type', '')).lower():
                    ctype = 'video'
                    break
    user = it.get('user') if isinstance(it.get('user'), dict) else {}
    row = {
        'platform': 'twitter',
        'content_id': str(it.get('tweet_id') or it.get('id') or ''),
        'user_id': None,
        'username': user.get('screen_name'),
        'author': user.get('name'),
        'title': text[:150],
        'description': text[:2000],
        'category': it.get('lang') or '',
        'tags': '',
        'tags_count': 0,
        'url': None,
        'content_type': ctype,
        'duration_sec': np.nan,
        'published_at': pub_iso,
        'post_hour': hour,
        'post_dayofweek': dow,
        'is_weekend': weekend,
        'is_vertical_reel': 0,
        'is_short_form': 1,
        'views': views,
        'likes': likes,
        'comments': comments,
        'shares': shares,
        'reactions_all': likes,
        'subscribers': lookup_subs('twitter'),
        'source': 'user_posts',
        'lang': it.get('lang'),
    }
    row.update(compute_metrics(row))
    rows_B.append(row)

df_B = pd.DataFrame(rows_B)
print('Model B complete:', df_B.shape)
print(df_B['platform'].value_counts())
print()
print(df_B[['platform', 'title', 'subscribers', 'views', 'breakout_log_ratio', 'engagement_score']].head(3))


Model B complete: (1381, 32)
platform
twitter      793
linkedin     276
facebook     200
youtube      100
instagram     12
Name: count, dtype: int64

    platform                                            title  subscribers  \
0  instagram                        Man Pregnant for 36 Years    3662163.0   
1  instagram  Israeli Spy Who Almost Became Syria’s President    3662163.0   
2  instagram                Why Jinnah Hated The Name INDIA ?    3662163.0   

       views  breakout_log_ratio  engagement_score  
0   731219.0           -1.611095         44.139125  
1   137814.0           -3.279897         76.670118  
2  1379774.0           -0.976134         42.312841  


---
## 6. Save CSVs

We keep a fixed column order so both files are easy to load in the training notebook.


In [16]:
cols_A = [
    'platform', 'content_id', 'title', 'description', 'category', 'tags', 'tags_count',
    'author', 'url', 'content_type', 'duration_sec',
    'published_at', 'post_hour', 'post_dayofweek', 'is_weekend',
    'is_vertical_reel', 'is_short_form',
    'views', 'likes', 'comments', 'shares', 'reactions_all', 'subscribers',
    'engagement_rate', 'quality_reaction_rate', 'breakout_log_ratio',
    'virality_index', 'engagement_score',
    'region', 'source', 'tweet_volume', 'trend_position',
]
cols_A = [c for c in cols_A if c in df_A.columns]
df_A = df_A[cols_A]

cols_B = [
    'platform', 'content_id', 'user_id', 'username', 'author',
    'title', 'description', 'category', 'tags', 'tags_count',
    'url', 'content_type', 'duration_sec',
    'published_at', 'post_hour', 'post_dayofweek', 'is_weekend',
    'is_vertical_reel', 'is_short_form',
    'views', 'likes', 'comments', 'shares', 'reactions_all', 'subscribers',
    'engagement_rate', 'quality_reaction_rate', 'breakout_log_ratio',
    'virality_index', 'engagement_score',
    'source', 'lang',
]
cols_B = [c for c in cols_B if c in df_B.columns]
df_B = df_B[cols_B]

path_A = OUT / 'model_A_trending.csv'
path_B = OUT / 'model_B_audience_posts.csv'
df_A.to_csv(path_A, index=False)
df_B.to_csv(path_B, index=False)

# Per-platform splits for Model B
for plat, g in df_B.groupby('platform'):
    g.to_csv(OUT / f'model_B_audience_{plat}.csv', index=False)

print('Saved:')
print(f'  {path_A.name:40s}  {len(df_A):5d} rows × {df_A.shape[1]} cols')
print(f'  {path_B.name:40s}  {len(df_B):5d} rows × {df_B.shape[1]} cols')
for p in sorted(OUT.glob('model_B_audience_*.csv')):
    if p.name != 'model_B_audience_posts.csv':
        print(f'  {p.name:40s}  {len(pd.read_csv(p)):5d} rows')


Saved:
  model_A_trending.csv                        153 rows × 32 cols
  model_B_audience_posts.csv                 1381 rows × 32 cols
  model_B_audience_facebook.csv               200 rows
  model_B_audience_instagram.csv               12 rows
  model_B_audience_linkedin.csv               276 rows
  model_B_audience_twitter.csv                793 rows
  model_B_audience_youtube.csv                100 rows


## 7. Quick look at the results

Check that the important engineered columns exist and look reasonable.


In [17]:
print('=== Model A (trending) ===')
print(df_A['platform'].value_counts())
print()
print(df_A[['title', 'content_type', 'views', 'likes',
            'breakout_log_ratio', 'quality_reaction_rate',
            'virality_index', 'engagement_score']].describe().round(2))

print('\n=== Model B (audience posts) ===')
print(df_B['platform'].value_counts())
print()
print(df_B.groupby('platform')[['subscribers', 'engagement_score', 'breakout_log_ratio']].mean().round(2))


=== Model A (trending) ===
platform
twitter      50
youtube      48
instagram    36
facebook     10
linkedin      9
Name: count, dtype: int64

             views      likes  breakout_log_ratio  quality_reaction_rate  \
count       153.00     153.00              153.00                 103.00   
mean    1935077.94   32244.20                9.12                   0.03   
std     3885576.26   89555.72                6.60                   0.04   
min           0.00       0.00                0.00                   0.00   
25%           0.00       0.00                0.00                   0.00   
50%      259312.00       0.00               12.47                   0.02   
75%     1680000.00    9986.00               14.33                   0.05   
max    19699762.00  621233.00               16.80                   0.18   

       virality_index  engagement_score  
count          153.00            153.00  
mean       2874501.76             27.79  
std        6157568.55             19.00  
min 

In [18]:
# One example row from each
print('--- Model A example (YouTube) ---')
ex = df_A[df_A['platform'] == 'youtube'].iloc[0]
for c in ['title', 'content_type', 'duration_sec', 'views', 'likes',
          'breakout_log_ratio', 'quality_reaction_rate', 'virality_index', 'engagement_score']:
    print(f'  {c:25s} {ex.get(c)}')

print('\n--- Model B example (Instagram) ---')
ex = df_B[df_B['platform'] == 'instagram'].iloc[0]
for c in ['title', 'subscribers', 'views', 'breakout_log_ratio',
          'virality_index', 'engagement_score', 'content_type']:
    print(f'  {c:25s} {ex.get(c)}')


--- Model A example (YouTube) ---
  title                     295 (MUSIC VIDEO) SIDHU MOOSE WALA
  content_type              video
  duration_sec              357.0
  views                     12183039.0
  likes                     621233.0
  breakout_log_ratio        16.31555537859472
  quality_reaction_rate     0.1093507949863741
  virality_index            25989484.0
  engagement_score          63.62749143389364

--- Model B example (Instagram) ---
  title                     Man Pregnant for 36 Years
  subscribers               3662163.0
  views                     731219.0
  breakout_log_ratio        -1.6110951364390946
  virality_index            1448849.0
  engagement_score          44.13912462601056
  content_type              video


---
## 8. What to do next

1. You now have:
   - `ml_ready/model_A_trending.csv`
   - `ml_ready/model_B_audience_posts.csv`
2. Open the **training notebook**  
   `conjugate-media-model-A-B-training.ipynb`  
   and run it to train Model A + Model B.
3. Or load the CSVs yourself:

```python
df_A = pd.read_csv('conjugate-media/server/data/ml_ready/model_A_trending.csv')
df_B = pd.read_csv('conjugate-media/server/data/ml_ready/model_B_audience_posts.csv')
```

Re-run **this** notebook anytime the raw JSON files change.
